In [3]:
import pandas as pd
import numpy as np
import sqlite3

# 1. SQL Database कनेक्शन बनाना (In-Memory SQL)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. SQL में एक 'operations' टेबल बनाना
cursor.execute('''
CREATE TABLE operations (
    ticket_id TEXT,
    process TEXT,
    hours_open TEXT,
    region TEXT
)
''')

# 3. SQL टेबल में डेटा भरना (इसमें "Forty" और "None" जैसा खराब डेटा भी शामिल है)
raw_data = [
    ("T-101", "Data Ops", "12", "IN"),
    ("T-102", "Quality Audit", "52", "US"),
    ("T-103", "Data Ops", "Forty", "IN"),
    ("T-104", "Reporting", "None", "UK"),
    ("T-105", "Analytics", "36", "IN")
]
cursor.executemany("INSERT INTO operations VALUES (?, ?, ?, ?)", raw_data)
conn.commit()

# =====================================================================
# PANDAS & ROBUST ERROR HANDLING (SQL से डेटा खींचना और क्लीन करना)
# =====================================================================
df = pd.read_sql_query("SELECT * FROM operations", conn)

print("=== 1. Raw Data Loaded From SQL Database ===")
print(df)

# ROBUST ERROR HANDLING: गंदे डेटा ("Forty", "None") को नंबर में बदलना, एरर आने पर NaN करना
df['hours_open'] = pd.to_numeric(df['hours_open'], errors='coerce')

# DATA CLEANING (fillna): खाली जगहों को कॉलम के Average/Mean से भरना
mean_hours = df['hours_open'].mean()
df['hours_open'] = df['hours_open'].fillna(mean_hours)

# DATA TRANSFORMATION (np.where): SLA स्टेटस मैप करना
df['SLA_Status'] = np.where(df['hours_open'] > 24, "🔴 Breached", "🟢 Within SLA")

print("\n=== 2. Final Cleaned & Transformed Analytics Report ===")
print(df)

# METRICS AGGREGATION (.groupby): रीज़न के हिसाब से टिकट काउंट देखना
print("\n=== 3. Executive Summary (Tickets Per Region) ===")
print(df.groupby('region')['ticket_id'].count())

conn.close()


=== 1. Raw Data Loaded From SQL Database ===
  ticket_id        process hours_open region
0     T-101       Data Ops         12     IN
1     T-102  Quality Audit         52     US
2     T-103       Data Ops      Forty     IN
3     T-104      Reporting       None     UK
4     T-105      Analytics         36     IN

=== 2. Final Cleaned & Transformed Analytics Report ===
  ticket_id        process  hours_open region    SLA_Status
0     T-101       Data Ops   12.000000     IN  🟢 Within SLA
1     T-102  Quality Audit   52.000000     US    🔴 Breached
2     T-103       Data Ops   33.333333     IN    🔴 Breached
3     T-104      Reporting   33.333333     UK    🔴 Breached
4     T-105      Analytics   36.000000     IN    🔴 Breached

=== 3. Executive Summary (Tickets Per Region) ===
region
IN    3
UK    1
US    1
Name: ticket_id, dtype: int64
